In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances

# Load preprocessed datasets
experience_metrics_with_clusters = pd.read_csv('../data/processed/experience_metrics_with_clusters.csv', index_col='MSISDN/Number')
normalized_experience_data = pd.read_csv('../data/processed/normalized_experience_data.csv', index_col='MSISDN/Number')

# Extract normalized data and cluster labels
X_normalized = normalized_experience_data.values
cluster_labels = experience_metrics_with_clusters['Experience Cluster']

# Re-run K-Means clustering (k=3) to get centroids
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_normalized)

# Identify the least engaged cluster (Cluster 0) and worst experience cluster (Cluster 0)
least_engaged_centroid = kmeans.cluster_centers_[0]  # Assuming Cluster 0 is the least engaged
worst_experience_centroid = kmeans.cluster_centers_[0]  # Assuming Cluster 0 is the worst experience

# Compute engagement and experience scores
engagement_scores = euclidean_distances(X_normalized, [least_engaged_centroid]).flatten()
experience_scores = euclidean_distances(X_normalized, [worst_experience_centroid]).flatten()

# Add scores to the dataset
experience_metrics_with_clusters['Engagement Score'] = engagement_scores
experience_metrics_with_clusters['Experience Score'] = experience_scores

# Display results
print("Engagement and Experience Scores Added:")
display(experience_metrics_with_clusters.head())

Engagement and Experience Scores Added:


,Avg TCP Retransmission (DL),Avg TCP Retransmission (UL),Avg RTT DL (ms),Avg RTT UL (ms),Handset Type,Avg Throughput DL (kbps),Avg Throughput UL (kbps),Avg Total Throughput (kbps),Experience Cluster,Engagement Score,Experience Score
MSISDN/Number,,,,,,,,,,,
3.360100e+10,2.080991e+07,759658.664811,46.000000,0.000000,Huawei P20 Lite Huawei Nova 3E,37.0,39.0,76.0,1,0.295333,0.295333
3.360100e+10,2.080991e+07,759658.664811,30.000000,1.000000,Apple iPhone 7 (A1778),48.0,51.0,99.0,1,0.295252,0.295252
3.360100e+10,2.080991e+07,759658.664811,109.795706,17.662883,undefined,48.0,49.0,97.0,1,0.295259,0.295259
3.360101e+10,1.066000e+03,759658.664811,69.000000,15.000000,Apple iPhone 5S (A1457),204.0,44.0,248.0,1,0.294967,0.294967
3.360101e+10,1.507977e+07,390430.332406,57.000000,2.500000,Apple iPhone Se (A1723),20197.5,8224.5,28422.0,2,0.195713,0.195713


In [3]:
# Compute satisfaction score
experience_metrics_with_clusters['Satisfaction Score'] = (
    experience_metrics_with_clusters['Engagement Score'] + experience_metrics_with_clusters['Experience Score']
) / 2

# Sort by satisfaction score and report top 10 satisfied customers
top_10_satisfied_customers = experience_metrics_with_clusters.sort_values(by='Satisfaction Score', ascending=False).head(10)

# Display results
print("Top 10 Satisfied Customers:")
display(top_10_satisfied_customers[['Engagement Score', 'Experience Score', 'Satisfaction Score']])

Top 10 Satisfied Customers:


,Engagement Score,Experience Score,Satisfaction Score
MSISDN/Number,,,
3.366232e+10,1.042090,1.042090,1.042090
3.369858e+10,1.000015,1.000015,1.000015
3.365871e+10,0.983378,0.983378,0.983378
3.366491e+10,0.983324,0.983324,0.983324
3.365863e+10,0.981933,0.981933,0.981933
3.366613e+10,0.968561,0.968561,0.968561
3.366877e+10,0.955260,0.955260,0.955260
3.366131e+10,0.949027,0.949027,0.949027
3.366240e+10,0.944723,0.944723,0.944723


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Select features and target
X = normalized_experience_data.values
y = experience_metrics_with_clusters['Satisfaction Score'].values

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict satisfaction scores
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Display results
print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared: {r2:.4f}")

Mean Squared Error: 0.0009
R-squared: 0.8226


In [5]:
# Select engagement and experience scores
scores_data = experience_metrics_with_clusters[['Engagement Score', 'Experience Score']].values

# Run K-Means clustering (k=2)
kmeans_satisfaction = KMeans(n_clusters=2, random_state=42)
experience_metrics_with_clusters['Satisfaction Cluster'] = kmeans_satisfaction.fit_predict(scores_data)

# Display cluster assignments
print("Satisfaction Clusters Assigned:")
display(experience_metrics_with_clusters[['Engagement Score', 'Experience Score', 'Satisfaction Cluster']].head())

Satisfaction Clusters Assigned:


,Engagement Score,Experience Score,Satisfaction Cluster
MSISDN/Number,,,
3.360100e+10,0.295333,0.295333,1
3.360100e+10,0.295252,0.295252,1
3.360100e+10,0.295259,0.295259,1
3.360101e+10,0.294967,0.294967,1
3.360101e+10,0.195713,0.195713,0


In [6]:
# Aggregate metrics per satisfaction cluster
satisfaction_summary = experience_metrics_with_clusters.groupby('Satisfaction Cluster').agg({
    'Engagement Score': ['mean', 'std'],
    'Experience Score': ['mean', 'std'],
    'Satisfaction Score': ['mean', 'std']
})

# Display results
print("Satisfaction Summary Per Cluster:")
display(satisfaction_summary)

Satisfaction Summary Per Cluster:


Engagement Score           Experience Score            \
                                 mean       std             mean       std   
Satisfaction Cluster                                                         
0                            0.134461  0.055278         0.134461  0.055278   
1                            0.284371  0.026357         0.284371  0.026357   

                     Satisfaction Score            
                                   mean       std  
Satisfaction Cluster                               
0                              0.134461  0.055278  
1                              0.284371  0.026357

In [13]:
%pip install sqlalchemy
import pandas as pd
import psycopg2
from sqlalchemy import create_engine

# Define PostgreSQL connection parameters
db_config = {
    'dbname': 'telecom',  # Replace with your database name
    'user': 'postgres',   # Replace with your username
    'password': 'your_password',  # Replace with your password
    'host': 'localhost',  # Replace with your host
    'port': 5432          # Default PostgreSQL port
}

# Create a connection engine for PostgreSQL
engine = create_engine(f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['dbname']}")

# Export the final table to PostgreSQL
final_table = experience_metrics_with_clusters[
    ['Engagement Score', 'Experience Score', 'Satisfaction Score', 'Satisfaction Cluster']
].reset_index()  # Include MSISDN/Number as user ID

final_table.to_sql(name='customer_satisfaction', con=engine, if_exists='replace', index=False)

print("Data exported to PostgreSQL database.")

Defaulting to user installation because normal site-packages is not writeable
  Using cached typing_extensions-4.12.2-py3-none-any.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.1 MB 2.1 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 2.4 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 2.5 MB/s eta 0:00:00
Using cached typing_extensions-4.12.2-py3-none-any.whl (37 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\Hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "postgres"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [14]:
import pandas as pd
from sqlalchemy import create_engine

# Define PostgreSQL connection parameters
db_config = {
    'dbname': 'telecom',  # Replace with your database name
    'user': 'postgres',   # Replace with your username
    'password': 'postgres',  # Replace with the actual password
    'host': 'localhost',  # Replace with your host (default is localhost)
    'port': 5432          # Default PostgreSQL port
}

# Create a connection engine for PostgreSQL
engine = create_engine(
    f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['dbname']}"
)

# Export the final table to PostgreSQL
final_table = experience_metrics_with_clusters[
    ['Engagement Score', 'Experience Score', 'Satisfaction Score', 'Satisfaction Cluster']
].reset_index()  # Include MSISDN/Number as user ID

try:
    final_table.to_sql(name='customer_satisfaction', con=engine, if_exists='replace', index=False)
    print("Data exported to PostgreSQL database.")
except Exception as e:
    print(f"Error exporting data: {e}")

Data exported to PostgreSQL database.
